# Qwen3 Sinhala QA — corrected evaluation and answer repair

Standalone evaluation for `isji/qwen3-4b-sinhala-qa-cpt-v2-merged` on the
external Sinhala QA split.

The generic Llama evaluator cannot be reused by changing only `MODEL_ID`: it sends a
SinLlama raw-text prompt, while this Qwen checkpoint was fine-tuned with Qwen chat
roles, `enable_thinking=False`, English `Context/Question/Answer` labels, and
`<|im_end|>` as the supervised answer terminator. That mismatch makes the CPT model
resume document/article text instead of completing a short QA turn.

This notebook fixes the inference contract and keeps three predictions for every row:

1. **Raw** — the complete decoded Qwen answer, before cleanup.
2. **Repaired** — label/tail/repetition cleanup plus conservative grounded-prefix
   trimming.
3. **Final** — the repaired answer after the configured safety policy. By default,
   grounding can reject only an unhealthy/non-terminating output; clean chat-EOS
   answers are preserved.

It also records EOS emission, token-cap hits, trims, gates, and raw/repaired/final
metrics. This makes inference-time mitigation distinguishable from genuine model
improvement.


In [ ]:
# Modal notebooks use %uv. In Colab/Jupyter, replace `%uv pip` with `%pip`.
%uv pip install -q "transformers>=4.57,<5" accelerate safetensors huggingface_hub hf_transfer pandas tqdm


In [ ]:
import hashlib
import json
import os
import re
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import set_seed

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ----------------------------- model and data -----------------------------
MODEL_ID = os.environ.get(
    "QA_MODEL_ID",
    "isji/qwen3-4b-sinhala-qa-cpt-v2-merged",
)

# Set QA_TEST_PATH when more than one candidate exists. Refusing ambiguity prevents
# a stale /tmp file from silently replacing the repository test split.
TEST_PATH_OVERRIDE = os.environ.get("QA_TEST_PATH")
TEST_CANDIDATES = [
    Path("new_split_v2/test.jsonl"),
    Path("../new_split_v2/test.jsonl"),
    Path("/tmp/test.jsonl"),
    Path("/tmp/test_updated.jsonl"),
]


def select_test_path():
    if TEST_PATH_OVERRIDE:
        selected = Path(TEST_PATH_OVERRIDE).expanduser().resolve()
        if not selected.is_file():
            raise FileNotFoundError(f"QA_TEST_PATH does not exist: {selected}")
        return selected

    found = []
    seen = set()
    for candidate in TEST_CANDIDATES:
        if candidate.is_file():
            resolved = candidate.resolve()
            if resolved not in seen:
                found.append(resolved)
                seen.add(resolved)
    if not found:
        raise FileNotFoundError(
            "No test JSONL found. Set QA_TEST_PATH to the intended file."
        )
    if len(found) > 1:
        choices = "\n".join(f"  - {path}" for path in found)
        raise RuntimeError(
            "Multiple test files exist; refusing to choose silently. Set QA_TEST_PATH.\n"
            + choices
        )
    return found[0]


TEST_PATH = select_test_path()
OUTPUT_DIR_OVERRIDE = os.environ.get("QA_OUTPUT_DIR")
RUN_ID = os.environ.get("QA_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = (
    Path(OUTPUT_DIR_OVERRIDE).expanduser()
    if OUTPUT_DIR_OVERRIDE
    else Path("/tmp/qwen3_sinhala_qa_fixed_eval") / RUN_ID
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
RESULTS_JSONL = OUTPUT_DIR / "predictions.jsonl"
RESULTS_TXT = OUTPUT_DIR / "results.txt"
RUNTIME_CONFIG_JSON = OUTPUT_DIR / "runtime-config.json"
SUMMARY_JSON = OUTPUT_DIR / "summary.json"

# ----------------------------- task contract ------------------------------
NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."
EXPECTED_VOCAB_SIZE = 176_856
EXPECTED_CHAT_EOS_ID = 151_645       # <|im_end|>, supervised by QA fine-tuning
EXPECTED_PAD_ID = 151_643            # <|endoftext|>, tracked as document/PAD contamination
STRICT_TOKENIZER_CHECKS = True

# These are fixed before looking at test labels. They match the Qwen-specific path.
MAX_SEQUENCE_TOKENS = 2_048
MAX_NEW_TOKENS = 64                  # train completion audit: maximum was 55 incl. EOS
NUM_BEAMS = 4                        # comparable path; use 6 only as a labeled sensitivity run
LENGTH_PENALTY = 1.0
REPETITION_PENALTY = 1.05
NO_REPEAT_NGRAM_SIZE = 0             # set to 4 only for a repetition sensitivity run

# Safety policy: "off", "unhealthy_only", or "always". The default avoids turning
# clean EOS-terminated answers into refusals because of a heuristic lexical score.
GROUNDING_POLICY = "unhealthy_only"
if GROUNDING_POLICY not in {"off", "unhealthy_only", "always"}:
    raise ValueError(f"Unknown GROUNDING_POLICY: {GROUNDING_POLICY}")
GROUNDING_THRESHOLD = 0.40           # same support semantics as the corrected Qwen path
ENABLE_GROUNDED_PREFIX_REPAIR = True
UNGROUNDED_RUN_TO_CUT = 3
REPEATED_NGRAM_SIZE = 5
MAX_OUTPUT_WORDS = 80

SEED = 42
RUN_SMOKE_TESTS = True
PRINT_EACH_ITEM = True

set_seed(SEED)
torch.manual_seed(SEED)

print("Model       :", MODEL_ID)
print("Test file   :", TEST_PATH)
print("Output dir  :", OUTPUT_DIR)
print("Decoding    :", f"beam x{NUM_BEAMS}" if NUM_BEAMS > 1 else "greedy")
print("Safety      :", f"{GROUNDING_POLICY} (threshold={GROUNDING_THRESHOLD})")


In [ ]:
# Only needed for private repositories. Never paste a token into the notebook.
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face from HF_TOKEN.")
else:
    print("HF_TOKEN is not set; continuing anonymously (works for public repositories).")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for the 4B evaluation notebook.")

model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading tokenizer from the merged QA repository: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = "<|endoftext|>"
tokenizer.padding_side = "left"

if not tokenizer.chat_template:
    raise RuntimeError("Tokenizer has no Qwen chat template; this is the wrong tokenizer.")

template_supports_thinking = "enable_thinking" in tokenizer.chat_template
CHAT_TEMPLATE_KWARGS = {"enable_thinking": False} if template_supports_thinking else {}

if STRICT_TOKENIZER_CHECKS:
    problems = []
    if len(tokenizer) != EXPECTED_VOCAB_SIZE:
        problems.append(
            f"vocabulary is {len(tokenizer):,}, expected {EXPECTED_VOCAB_SIZE:,}"
        )
    if tokenizer.eos_token_id != EXPECTED_CHAT_EOS_ID:
        problems.append(
            f"EOS is {tokenizer.eos_token!r} ({tokenizer.eos_token_id}), "
            f"expected <|im_end|> ({EXPECTED_CHAT_EOS_ID})"
        )
    if tokenizer.pad_token_id != EXPECTED_PAD_ID:
        problems.append(
            f"PAD is {tokenizer.pad_token!r} ({tokenizer.pad_token_id}), "
            f"expected <|endoftext|> ({EXPECTED_PAD_ID})"
        )
    if problems:
        raise RuntimeError("Tokenizer/model contract mismatch:\n- " + "\n- ".join(problems))

print(f"Loading model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=model_dtype,
    device_map="auto",
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = True

input_weight = model.get_input_embeddings().weight
output_weight = model.get_output_embeddings().weight
embedding_rows = input_weight.shape[0]
output_rows = output_weight.shape[0]
if embedding_rows != len(tokenizer) or output_rows != len(tokenizer):
    raise RuntimeError(
        "Merged model/tokenizer mismatch: "
        f"input={embedding_rows:,}, output={output_rows:,}, tokenizer={len(tokenizer):,}. "
        "Load the tokenizer saved with the merged model."
    )
if not getattr(model.config, "tie_word_embeddings", False):
    raise RuntimeError(
        "This CPT+QA artifact requires tie_word_embeddings=True. An untied lm_head can carry "
        "stale rows for the 25,187 added Sinhala tokens."
    )
if input_weight.data_ptr() != output_weight.data_ptr():
    raise RuntimeError(
        "The merged checkpoint declares tied embeddings, but embed_tokens and lm_head are "
        "not tied. The added Sinhala tokens may be ungeneratable."
    )

# Clear sampling fields inherited from generation_config.json. All reported runs are
# deterministic. Stop only on the supervised chat EOS. Treating the document separator
# as EOS would mask the exact QA-termination failure this notebook is designed to audit.
model.generation_config.do_sample = False
for name in ("temperature", "top_p", "top_k", "min_p"):
    if hasattr(model.generation_config, name):
        setattr(model.generation_config, name, None)
model.generation_config.pad_token_id = tokenizer.pad_token_id

EOS_TOKEN_IDS = [EXPECTED_CHAT_EOS_ID]
DOCUMENT_END_ID = EXPECTED_PAD_ID
model.generation_config.eos_token_id = EXPECTED_CHAT_EOS_ID
model_commit = getattr(model.config, "_commit_hash", None)
tokenizer_commit = tokenizer.init_kwargs.get("_commit_hash")

print("Loaded model/tokenizer contract")
print("  vocabulary       :", f"{len(tokenizer):,}")
print("  embeddings       :", f"{embedding_rows:,}")
print("  dtype            :", model_dtype)
print("  chat EOS ID      :", EXPECTED_CHAT_EOS_ID)
print("  document/PAD ID  :", DOCUMENT_END_ID)
print("  thinking switch  :", "present -> disabled" if template_supports_thinking else "not present")


In [ ]:
# This text and message layout intentionally match qwen3-4b-sinhala-qa-on-cpt.ipynb.
SYSTEM_PROMPT = f"""You are a helpful Sinhala history question-answering assistant.

Your task is to answer the question using ONLY the information explicitly provided in the context.

Instructions:

- Read the entire context carefully before answering.
- Use only the information explicitly stated in the context.
- Do not use external knowledge, assumptions, or prior knowledge.
- Identify the exact information requested by the question.
- If the answer is found in multiple parts of the context, combine the relevant information into a single complete answer.
- Include only information that directly answers the question.
- Do not include additional facts, names, dates, or events unless they are required to answer the question.
- Match the person or entity named in the question exactly.
- Use evidence that contains both the requested entity and the requested attribute.
- Do not take a date or fact from a neighboring sentence about a different entity or event.
- Do not infer or guess information that is not explicitly stated, except for simple arithmetic explicitly requested by the question when all required values are stated in the context.
- For a duration question with explicit starting and ending years, subtract the starting year from the ending year and return the duration.
- If the answer cannot be found in the context, respond exactly with:
  "{NO_ANSWER}"
- Return only the final answer in natural Sinhala.
- Do not explain your reasoning.
- Do not mention passage numbers, page numbers, chapter names, grades, or any other source references.
- Answer in a single line, then stop. Do not continue with any further text."""


def clean_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def build_messages(context, question):
    user_prompt = f"""Context:

{clean_text(context)}

Question:

{clean_text(question)}

Answer:"""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def render_prompt(context, question):
    return tokenizer.apply_chat_template(
        build_messages(context, question),
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_TEMPLATE_KWARGS,
    )


def encode_prompt(context, question):
    # The chat template already inserted every special token. Adding them again is a
    # prompt-contract bug, so add_special_tokens must remain False.
    prompt = render_prompt(context, question)
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    return prompt, encoded


print("Qwen training-matched prompt builder ready.")


In [ ]:
SINHALA_WORD_RE = re.compile(r"[\w\u0D80-\u0DFF]+", re.UNICODE)
STOPWORDS = {
    "හා", "සහ", "හෝ", "දී", "ද", "ය", "යි", "වේ", "විය", "වූ", "ලෙස",
    "විසින්", "සඳහා", "සිට", "දක්වා", "එම", "මෙම", "ඒ", "ඔහු", "ඇය",
    "කුමක්ද", "කවුද", "කවදාද", "කෙසේද", "කොපමණද", "මොනවාද",
}
REFUSAL_MARKERS = (
    "ප්‍රමාණවත් තොරතුරු නොමැත",
    "පිළිතුර නොමැත",
    "පිළිතුරු නොමැත",
    "සඳහන් වී නොමැත",
    "සඳහන් නොවේ",
    "දක්වා නොමැත",
    "not enough information",
    "no answer",
    "cannot be found",
)


def lexical_tokens(value):
    tokens = [token.casefold() for token in SINHALA_WORD_RE.findall(clean_text(value))]
    return [token for token in tokens if len(token) >= 2 and token not in STOPWORDS]


def token_supported(token, normalized_context):
    """Qwen-path lexical support with a digit-boundary guard.

    This intentionally keeps the score distribution used with the frozen 0.40 threshold.
    It is a provenance heuristic, not a semantic relevance check.
    """
    if token.isdigit():
        return bool(re.search(rf"(?<!\d){re.escape(token)}(?!\d)", normalized_context))
    if token in normalized_context:
        return True
    return len(token) >= 4 and token[:-1] in normalized_context


def evidence_support(answer, context):
    answer_tokens = lexical_tokens(answer)
    if not answer_tokens:
        return 0.0
    normalized_context = " ".join(lexical_tokens(context))
    supported = sum(token_supported(token, normalized_context) for token in answer_tokens)
    return supported / len(answer_tokens)


def normalize_answer(value):
    text = clean_text(value).casefold()
    text = re.sub(r"\s+", " ", text)
    # Keep ZWJ (U+200D): it is valid Sinhala shaping, not encoding damage.
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴*")


def is_no_answer(value):
    normalized = normalize_answer(value)
    if not normalized:
        return False
    if normalized == normalize_answer(NO_ANSWER):
        return True
    # Do not classify a correct answer plus a later refusal-like phrase as an abstention.
    # Alternate refusals must themselves begin with the refusal marker.
    for marker in REFUSAL_MARKERS:
        marker_normalized = normalize_answer(marker)
        if normalized == marker_normalized or normalized.startswith(marker_normalized + " "):
            return True
    return False


COPULA_SUFFIXES = (
    " ලෙස ය", " යනුවෙනි", " වශයෙනි", " ලෙසිනි", " විසිනි",
    " වේ", " යි", " ය", "යනුවෙනි", "වශයෙනි", "ලෙසිනි", "යි",
)


def strip_copula(value):
    text = normalize_answer(value)
    text = re.sub(r"\s+යි$", "යි", text)
    changed = True
    while changed:
        changed = False
        for suffix in COPULA_SUFFIXES:
            if text.endswith(suffix) and len(text) > len(suffix):
                text = text[:-len(suffix)].strip(" ,.।")
                changed = True
                break
    return text


def style_insensitive_match(prediction, reference):
    return strip_copula(prediction) == strip_copula(reference)


def token_f1(prediction, reference, guard_digits=True):
    prediction_digits = re.findall(r"\d+", normalize_answer(prediction))
    reference_digits = re.findall(r"\d+", normalize_answer(reference))
    if guard_digits and reference_digits and prediction_digits != reference_digits:
        return 0.0
    prediction_tokens = lexical_tokens(prediction)
    reference_tokens = lexical_tokens(reference)
    if not prediction_tokens and not reference_tokens:
        return 1.0
    if not prediction_tokens or not reference_tokens:
        return 0.0
    overlap = sum((Counter(prediction_tokens) & Counter(reference_tokens)).values())
    if not overlap:
        return 0.0
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


def score_prediction(prediction, reference):
    return {
        "exact": normalize_answer(prediction) == normalize_answer(reference),
        "style_exact": style_insensitive_match(prediction, reference),
        "guarded_f1": token_f1(prediction, reference, guard_digits=True),
        "relaxed_f1": token_f1(prediction, reference, guard_digits=False),
    }


def canonical_answer(item):
    if item.get("answerable") is False:
        return NO_ANSWER
    answer = clean_text(item.get("answer", ""))
    return answer if answer else NO_ANSWER


def load_jsonl(path):
    records, fingerprints = [], set()
    dropped = duplicates = 0
    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error

            question = clean_text(item.get("question"))
            context = clean_text(item.get("context"))
            answerable = item.get("answerable")
            if type(answerable) is not bool:
                answerable = bool(clean_text(item.get("answer")))

            normalized = {
                "question": question,
                "context": context,
                "answer": clean_text(item.get("answer")),
                "answerable": answerable,
                "grade": item.get("grade"),
                "chapter": item.get("chapter"),
                "chapter_title": item.get("chapter_title"),
            }
            if not question or not context or (answerable and not normalized["answer"]):
                dropped += 1
                continue
            fingerprint = (question, context, normalized["answer"], answerable)
            if fingerprint in fingerprints:
                duplicates += 1
                continue
            fingerprints.add(fingerprint)
            records.append(normalized)

    print(f"Loaded {len(records)} unique rows from {path}")
    print(f"Dropped invalid/empty: {dropped}; exact duplicates removed: {duplicates}")
    return records


print("Data, grounding, and scoring helpers ready.")


In [ ]:
test_records = load_jsonl(TEST_PATH)
if not test_records:
    raise RuntimeError("The selected test dataset is empty after validation.")

dataset_sha256 = hashlib.sha256(TEST_PATH.read_bytes()).hexdigest()
print("Answerability :", Counter(row["answerable"] for row in test_records))
print("SHA256        :", dataset_sha256)

# Fail before inference if the prompt would leave the fine-tuning length regime.
prompt_lengths = []
gold_lengths = []
bad_unicode_rows = []
for index, row in enumerate(test_records, 1):
    prompt, encoded = encode_prompt(row["context"], row["question"])
    prompt_lengths.append(int(encoded["input_ids"].shape[-1]))
    gold_lengths.append(
        len(tokenizer(canonical_answer(row), add_special_tokens=False)["input_ids"])
    )
    values = (row["question"], row["context"], canonical_answer(row))
    if any("\ufffd" in value or value != unicodedata.normalize("NFC", value) for value in values):
        bad_unicode_rows.append(index)

if bad_unicode_rows:
    raise RuntimeError(f"Encoding/NFC problems in rows: {bad_unicode_rows[:20]}")

def percentile(values, fraction):
    ordered = sorted(values)
    return ordered[min(len(ordered) - 1, int(len(ordered) * fraction))]


print(
    "Prompt tokens :",
    f"median {percentile(prompt_lengths, 0.50)}, "
    f"p95 {percentile(prompt_lengths, 0.95)}, max {max(prompt_lengths)}",
)
print(
    "Gold tokens   :",
    f"median {percentile(gold_lengths, 0.50)}, "
    f"p95 {percentile(gold_lengths, 0.95)}, max {max(gold_lengths)} "
    f"(generation cap {MAX_NEW_TOKENS})",
)

over_budget = [
    (index, length)
    for index, length in enumerate(prompt_lengths, 1)
    if length + MAX_NEW_TOKENS > MAX_SEQUENCE_TOKENS
]
if over_budget:
    raise RuntimeError(
        f"{len(over_budget)} rows exceed the {MAX_SEQUENCE_TOKENS}-token QA regime; "
        "retrieve a shorter evidence passage instead of silently truncating. "
        f"First rows: {over_budget[:10]}"
    )
if max(gold_lengths) >= MAX_NEW_TOKENS:
    raise RuntimeError(
        f"Longest gold answer is {max(gold_lengths)} tokens, too close to "
        f"MAX_NEW_TOKENS={MAX_NEW_TOKENS}. Raise the cap."
    )

print(f"All {len(test_records)} prompts fit; no context will be truncated.")


In [ ]:
LABEL_PREFIX_RE = re.compile(r"^\s*(?:පිළිතුර|answer)\s*[:：]\s*", re.IGNORECASE)
ARTICLE_TAIL_MARKERS = (
    "මෙම ලිපිය",
    "මූලාශ්‍ර:",
    "Sources:",
    "References:",
    "<|im_start|>",
    "<|im_end|>",
    "<|endoftext|>",
)


def cut_repeated_ngram(text, n=REPEATED_NGRAM_SIZE):
    """Return (text, character cut position) at the second repeated word n-gram."""
    words = clean_text(text).split()
    if len(words) < 2 * n:
        return clean_text(text), None
    seen = {}
    normalized_words = [normalize_answer(word) for word in words]
    for index in range(len(words) - n + 1):
        ngram = tuple(normalized_words[index:index + n])
        if not all(ngram):
            continue
        if ngram in seen and index - seen[ngram] >= n:
            prefix = " ".join(words[:index]).strip()
            return prefix, len(prefix)
        seen.setdefault(ngram, index)
    cleaned = " ".join(words).strip()
    return cleaned, None


def structural_cleanup(text):
    """Remove explicit template/article/repetition tails and record every reason."""
    answer = clean_text(text).replace("\x00", "")
    reasons = []
    cut_position = None
    if not answer:
        return {"answer": "", "reasons": reasons, "cut_position": cut_position}

    think_match = re.match(r"(?is)^\s*<think>.*?</think>\s*", answer)
    if think_match:
        answer = answer[think_match.end():]
        reasons.append("thinking_block")

    label_match = LABEL_PREFIX_RE.match(answer)
    if label_match:
        answer = answer[label_match.end():]
        reasons.append("answer_label")

    lines = [line.strip() for line in answer.splitlines() if line.strip()]
    if len(lines) > 1:
        cut_position = len(lines[0])
        reasons.append("extra_lines")
    answer = lines[0] if lines else ""

    folded = answer.casefold()
    cuts = []
    for marker in ARTICLE_TAIL_MARKERS:
        position = folded.find(marker.casefold())
        if position >= 0:
            cuts.append((position, f"tail_marker:{marker}"))
    bracket = re.search(r"[\[【]", answer)
    if bracket:
        cuts.append((bracket.start(), "bracket_tail"))
    if cuts:
        marker_position, marker_reason = min(cuts, key=lambda value: value[0])
        answer = answer[:marker_position]
        cut_position = marker_position if cut_position is None else min(cut_position, marker_position)
        reasons.append(marker_reason)

    answer, repeated_cut = cut_repeated_ngram(answer)
    if repeated_cut is not None:
        cut_position = repeated_cut if cut_position is None else min(cut_position, repeated_cut)
        reasons.append("repeated_ngram")

    words = answer.split()
    if len(words) > MAX_OUTPUT_WORDS:
        answer = " ".join(words[:MAX_OUTPUT_WORDS])
        cut_position = len(answer) if cut_position is None else min(cut_position, len(answer))
        reasons.append("word_limit")

    return {
        "answer": answer.strip(" []{}()<>\"'`*"),
        "reasons": reasons,
        "cut_position": cut_position,
    }


def grounded_prefix(text, context, run=UNGROUNDED_RUN_TO_CUT):
    """Return (prefix, cut word index), but only after a grounded prefix exists."""
    answer = clean_text(text)
    if run <= 0 or not answer or is_no_answer(answer):
        return answer, None

    normalized_context = " ".join(lexical_tokens(context))
    words = answer.split()
    consecutive_bad = 0
    supported_content_seen = False
    for index, word in enumerate(words):
        tokens = lexical_tokens(word)
        supported = not tokens or any(
            token_supported(token, normalized_context) for token in tokens
        )
        if supported:
            if tokens:
                supported_content_seen = True
            consecutive_bad = 0
        else:
            consecutive_bad += 1
        if consecutive_bad >= run and supported_content_seen:
            cut_index = index - run + 1
            return " ".join(words[:cut_index]).strip(), cut_index
    return " ".join(words).strip(), None


def repair_answer(raw_answer, context, allow_grounded_prefix=False):
    structural = structural_cleanup(raw_answer)
    structural_answer = structural["answer"]
    reasons = list(structural["reasons"])
    grounding_cut = None

    if is_no_answer(structural_answer):
        repaired = NO_ANSWER
    elif (
        ENABLE_GROUNDED_PREFIX_REPAIR
        and (allow_grounded_prefix or bool(reasons))
    ):
        repaired, grounding_cut = grounded_prefix(structural_answer, context)
        if grounding_cut is not None:
            reasons.append("ungrounded_prefix")
    else:
        repaired = structural_answer

    return {
        "structural_answer": structural_answer,
        "repaired_answer": repaired,
        "repair_reasons": reasons,
        "repair_cut_position": structural["cut_position"],
        "structural_trimmed": bool(structural["reasons"]),
        "grounding_trimmed": grounding_cut is not None,
    }


# Synthetic invariants exercise the repair contract without inspecting external-test labels.
_fixture_answer = "නිවැරදි පිළිතුර ය."
_fixture_context = "මෙහි නිවැරදි පිළිතුර ය. තවත් වාක්‍යයක් ඇත."
_fixture_clean = repair_answer(
    _fixture_answer,
    _fixture_context,
    allow_grounded_prefix=False,
)
assert normalize_answer(_fixture_clean["repaired_answer"]) == normalize_answer(_fixture_answer)
assert repair_answer(NO_ANSWER, _fixture_context, False)["repaired_answer"] == NO_ANSWER
_fixture_tail = structural_cleanup(_fixture_answer + " මෙම ලිපිය අනවශ්‍ය පෙළකි.")
assert normalize_answer(_fixture_tail["answer"]) == normalize_answer(_fixture_answer)

print("Bounded output-repair helpers ready; synthetic invariants passed.")


In [ ]:
GENERATE_KWARGS = {
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": False,
    "repetition_penalty": REPETITION_PENALTY,
    "eos_token_id": EXPECTED_CHAT_EOS_ID,
    "pad_token_id": tokenizer.pad_token_id,
    "use_cache": True,
}
if NUM_BEAMS > 1:
    GENERATE_KWARGS.update(
        num_beams=NUM_BEAMS,
        length_penalty=LENGTH_PENALTY,
        early_stopping=True,
    )
if NO_REPEAT_NGRAM_SIZE > 0:
    GENERATE_KWARGS["no_repeat_ngram_size"] = NO_REPEAT_NGRAM_SIZE


def generate_raw(context, question):
    prompt, encoded = encode_prompt(context, question)
    prompt_tokens = int(encoded["input_ids"].shape[-1])
    if prompt_tokens + MAX_NEW_TOKENS > MAX_SEQUENCE_TOKENS:
        raise ValueError(
            f"Prompt ({prompt_tokens}) + output cap ({MAX_NEW_TOKENS}) exceeds "
            f"MAX_SEQUENCE_TOKENS={MAX_SEQUENCE_TOKENS}."
        )

    inputs = {name: tensor.to(model.device) for name, tensor in encoded.items()}
    with torch.inference_mode():
        output_ids = model.generate(**inputs, **GENERATE_KWARGS)

    generated = output_ids[0, prompt_tokens:]
    all_ids = generated.tolist()
    chat_eos_positions = [
        index for index, token_id in enumerate(all_ids)
        if token_id == EXPECTED_CHAT_EOS_ID
    ]
    if chat_eos_positions:
        first_stop_position = chat_eos_positions[0]
        effective_ids = all_ids[:first_stop_position + 1]
        emitted_chat_eos = True
    else:
        first_stop_position = None
        effective_ids = all_ids
        emitted_chat_eos = False

    pre_chat_eos_ids = (
        all_ids[:first_stop_position] if first_stop_position is not None else all_ids
    )
    document_end_positions = [
        index for index, token_id in enumerate(pre_chat_eos_ids)
        if token_id == DOCUMENT_END_ID
    ]
    emitted_document_end = bool(document_end_positions)
    document_end_position = document_end_positions[0] if document_end_positions else None
    repair_ids = (
        effective_ids[:document_end_position]
        if document_end_position is not None
        else effective_ids
    )

    # Preserve the full decoded model output for raw scoring/audit. Separately decode the
    # candidate before a document separator so skip_special_tokens cannot concatenate a
    # correct answer with the following CPT corpus continuation.
    raw_answer = tokenizer.decode(
        effective_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()
    pre_document_answer = tokenizer.decode(
        repair_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()
    diagnostic_decode = tokenizer.decode(
        effective_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    ).strip()
    hit_token_cap = not emitted_chat_eos and len(all_ids) >= MAX_NEW_TOKENS

    return {
        "raw_answer": raw_answer,
        "pre_document_answer": pre_document_answer,
        "diagnostic_decode": diagnostic_decode,
        "prompt_tokens": prompt_tokens,
        "generated_tokens": len(effective_ids),
        "emitted_eos": emitted_chat_eos,
        "stop_token_id": EXPECTED_CHAT_EOS_ID if emitted_chat_eos else None,
        "emitted_document_end": emitted_document_end,
        "document_end_position": document_end_position,
        "hit_token_cap": hit_token_cap,
    }


def format_support(value):
    return "n/a" if value is None else f"{value:.3f}"


def run_qa(context, question, grounding_policy=GROUNDING_POLICY):
    if grounding_policy not in {"off", "unhealthy_only", "always"}:
        raise ValueError(f"Unknown grounding policy: {grounding_policy}")

    generated = generate_raw(context, question)
    unhealthy_termination = (
        not generated["emitted_eos"]
        or generated["hit_token_cap"]
        or generated["emitted_document_end"]
    )
    repaired = repair_answer(
        generated["pre_document_answer"],
        context,
        allow_grounded_prefix=(
            unhealthy_termination or grounding_policy == "always"
        ),
    )
    if generated["emitted_document_end"]:
        repaired["repair_reasons"] = ["document_end"] + repaired["repair_reasons"]
        repaired["structural_trimmed"] = True
        if repaired["repair_cut_position"] is None:
            repaired["repair_cut_position"] = len(generated["pre_document_answer"])
    candidate = repaired["repaired_answer"]
    repair_signal = bool(repaired["repair_reasons"])
    unhealthy_output = unhealthy_termination or repair_signal

    empty_generation = not clean_text(generated["raw_answer"])
    direct_refusal = bool(candidate) and is_no_answer(candidate)
    gate_eligible = (
        grounding_policy == "always"
        or (grounding_policy == "unhealthy_only" and unhealthy_output)
    )

    if empty_generation:
        final_answer = NO_ANSWER
        support = None
        decision = "empty_generation"
        gated = False
    elif direct_refusal or not candidate:
        final_answer = NO_ANSWER
        support = None
        decision = "model_refusal" if direct_refusal else "empty_after_repair"
        gated = False
    else:
        support = evidence_support(candidate, context)
        if gate_eligible and support < GROUNDING_THRESHOLD:
            final_answer = NO_ANSWER
            decision = "grounding_gate"
            gated = True
        else:
            final_answer = candidate
            decision = "accepted"
            gated = False

    return {
        **generated,
        **repaired,
        "answer": final_answer,
        "evidence_support": support,
        "decision": decision,
        "gated": gated,
        "gate_eligible": gate_eligible,
        "unhealthy_termination": unhealthy_termination,
        "unhealthy_output": unhealthy_output,
        "empty_generation": empty_generation,
        "direct_refusal": direct_refusal,
        "candidate_verbatim_in_context": bool(candidate)
        and not is_no_answer(candidate)
        and normalize_answer(candidate) in normalize_answer(context),
    }


print("Generation and bounded safety path ready.")
print("Generation kwargs:", GENERATE_KWARGS)


In [ ]:
# Prompt-contract assertions: these deliberately replace the SinLlama assertions in
# llama-scripts/qa-evaluation.ipynb.
preview_row = test_records[0]
preview = render_prompt(preview_row["context"], preview_row["question"])
assert "<|im_start|>system" in preview, "Qwen system role is missing"
assert "<|im_start|>user" in preview, "Qwen user role is missing"
assert "<|im_start|>assistant" in preview, "Qwen generation role is missing"
assert "Context:\n\n" in preview and "Question:\n\n" in preview and "Answer:" in preview
assert "උපදෙස්: පහත සන්දර්භය" not in preview, "SinLlama prompt leaked into Qwen evaluation"
if template_supports_thinking:
    assert preview.endswith("<think>\n\n</think>\n\n"), (
        "enable_thinking=False did not render the training-matched no-thinking suffix"
    )

print("Prompt contract matches Qwen QA training.")
print("--- rendered prompt tail (repr) ---")
print(repr(preview[-700:]))

if RUN_SMOKE_TESTS:
    smoke_rows = []
    answerable_row = next((row for row in test_records if row["answerable"]), None)
    if answerable_row is not None:
        smoke_rows.append(answerable_row)
    unanswerable_row = next(
        (row for row in test_records if not row["answerable"]),
        None,
    )
    if unanswerable_row is not None:
        smoke_rows.append(unanswerable_row)

    print("\n--- smoke tests ---")
    for row in smoke_rows:
        result = run_qa(row["context"], row["question"])
        print("-" * 100)
        print("Answerable :", row["answerable"])
        print("Question   :", row["question"])
        print("Expected   :", canonical_answer(row))
        print("Raw        :", result["raw_answer"])
        print("Repaired   :", result["repaired_answer"])
        print("Final      :", result["answer"])
        print(
            "Stop/health :",
            f"eos={result['emitted_eos']} id={result['stop_token_id']} "
            f"tokens={result['generated_tokens']} cap={result['hit_token_cap']} "
            f"decision={result['decision']} "
            f"support={format_support(result['evidence_support'])}",
        )


In [ ]:
runtime_config = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "test_path": str(TEST_PATH),
    "test_sha256": dataset_sha256,
    "run_id": RUN_ID,
    "output_dir": str(OUTPUT_DIR),
    "model_commit": model_commit,
    "tokenizer_commit": tokenizer_commit,
    "gpu": torch.cuda.get_device_name(0),
    "model_dtype": str(model_dtype),
    "system_prompt_sha256": hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest(),
    "chat_template_sha256": hashlib.sha256(tokenizer.chat_template.encode("utf-8")).hexdigest(),
    "rows": len(test_records),
    "transformers_version": transformers.__version__,
    "torch_version": torch.__version__,
    "tokenizer_size": len(tokenizer),
    "embedding_rows": embedding_rows,
    "tokenizer_eos_token": tokenizer.eos_token,
    "tokenizer_eos_token_id": tokenizer.eos_token_id,
    "tokenizer_pad_token": tokenizer.pad_token,
    "tokenizer_pad_token_id": tokenizer.pad_token_id,
    "chat_eos_id": EXPECTED_CHAT_EOS_ID,
    "document_end_id": DOCUMENT_END_ID,
    "chat_template_supports_thinking": template_supports_thinking,
    "chat_template_kwargs": CHAT_TEMPLATE_KWARGS,
    "max_sequence_tokens": MAX_SEQUENCE_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "num_beams": NUM_BEAMS,
    "length_penalty": LENGTH_PENALTY,
    "repetition_penalty": REPETITION_PENALTY,
    "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
    "grounding_policy": GROUNDING_POLICY,
    "grounding_threshold": GROUNDING_THRESHOLD,
    "enable_grounded_prefix_repair": ENABLE_GROUNDED_PREFIX_REPAIR,
    "ungrounded_run_to_cut": UNGROUNDED_RUN_TO_CUT,
    "repeated_ngram_size": REPEATED_NGRAM_SIZE,
    "max_output_words": MAX_OUTPUT_WORDS,
    "article_tail_markers": ARTICLE_TAIL_MARKERS,
    "strict_tokenizer_checks": STRICT_TOKENIZER_CHECKS,
    "seed": SEED,
}
RUNTIME_CONFIG_JSON.write_text(
    json.dumps(runtime_config, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

INCOMPLETE_MARKER = OUTPUT_DIR / "RUN_INCOMPLETE"
COMPLETE_MARKER = OUTPUT_DIR / "RUN_COMPLETE.json"
INCOMPLETE_MARKER.write_text("evaluation started; summary not finalized\n", encoding="utf-8")

predictions = []
transcript_lines = [
    f"Model: {MODEL_ID}",
    f"Test: {TEST_PATH}",
    f"Test SHA256: {dataset_sha256}",
    f"Decoding: {'beam x' + str(NUM_BEAMS) if NUM_BEAMS > 1 else 'greedy'}",
]

with RESULTS_JSONL.open("w", encoding="utf-8", newline="\n") as results_file:
    for index, item in enumerate(tqdm(test_records, desc="Evaluating"), 1):
        reference = canonical_answer(item)
        result = run_qa(item["context"], item["question"])

        raw_scores = score_prediction(result["raw_answer"], reference)
        repaired_scores = score_prediction(result["repaired_answer"], reference)
        final_scores = score_prediction(result["answer"], reference)

        record = {
            "index": index,
            "grade": item.get("grade"),
            "chapter": item.get("chapter"),
            "chapter_title": item.get("chapter_title"),
            "answerable": item["answerable"],
            "question": item["question"],
            "context": item["context"],
            "reference": reference,
            "raw_prediction": result["raw_answer"],
            "pre_document_prediction": result["pre_document_answer"],
            "diagnostic_decode": result["diagnostic_decode"],
            "structural_prediction": result["structural_answer"],
            "repaired_prediction": result["repaired_answer"],
            "prediction": result["answer"],
            "raw_exact_match": raw_scores["exact"],
            "raw_style_exact_match": raw_scores["style_exact"],
            "raw_token_f1": raw_scores["guarded_f1"],
            "raw_relaxed_token_f1": raw_scores["relaxed_f1"],
            "repaired_exact_match": repaired_scores["exact"],
            "repaired_style_exact_match": repaired_scores["style_exact"],
            "repaired_token_f1": repaired_scores["guarded_f1"],
            "repaired_relaxed_token_f1": repaired_scores["relaxed_f1"],
            "exact_match": final_scores["exact"],
            "style_exact_match": final_scores["style_exact"],
            "token_f1": final_scores["guarded_f1"],
            "relaxed_token_f1": final_scores["relaxed_f1"],
            "evidence_support": result["evidence_support"],
            "decision": result["decision"],
            "gated": result["gated"],
            "structural_trimmed": result["structural_trimmed"],
            "grounding_trimmed": result["grounding_trimmed"],
            "empty_generation": result["empty_generation"],
            "direct_refusal": result["direct_refusal"],
            "candidate_verbatim_in_context": result["candidate_verbatim_in_context"],
            "repair_reasons": result["repair_reasons"],
            "repair_cut_position": result["repair_cut_position"],
            "gate_eligible": result["gate_eligible"],
            "unhealthy_termination": result["unhealthy_termination"],
            "unhealthy_output": result["unhealthy_output"],
            "emitted_eos": result["emitted_eos"],
            "emitted_document_end": result["emitted_document_end"],
            "document_end_position": result["document_end_position"],
            "stop_token_id": result["stop_token_id"],
            "hit_token_cap": result["hit_token_cap"],
            "prompt_tokens": result["prompt_tokens"],
            "generated_tokens": result["generated_tokens"],
        }
        predictions.append(record)
        results_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        results_file.flush()

        block = [
            "",
            "=" * 100,
            f"[{index}/{len(test_records)}]",
            f"Answerable: {item['answerable']}",
            f"Question : {item['question']}",
            f"Expected : {reference}",
            f"Raw      : {result['raw_answer']}",
            f"Repaired : {result['repaired_answer']}",
            f"Final    : {result['answer']}",
            f"Decision : {result['decision']} | support {format_support(result['evidence_support'])}",
            f"Repair   : {result['repair_reasons'] or ['none']}",
            f"EOS/doc/cap: {result['emitted_eos']} / {result['emitted_document_end']} / {result['hit_token_cap']}",
            f"Exact/F1 : {final_scores['exact']} / {final_scores['guarded_f1']:.3f}",
        ]
        transcript_lines.extend(block)
        if PRINT_EACH_ITEM:
            print("\n".join(block), flush=True)

print("\nFinished evaluation.")
print("Per-item JSONL:", RESULTS_JSONL)


In [ ]:
STAGES = {
    "raw": "raw_prediction",
    "repaired": "repaired_prediction",
    "final": "prediction",
}


def aggregate_stage(rows, prediction_field):
    scored = [score_prediction(row[prediction_field], row["reference"]) for row in rows]
    answerable_rows = [
        (row, score) for row, score in zip(rows, scored) if row["answerable"]
    ]
    unanswerable_rows = [
        (row, score) for row, score in zip(rows, scored) if not row["answerable"]
    ]

    def predicted_refusal(value):
        return bool(normalize_answer(value)) and is_no_answer(value)

    refusal_predictions = [
        row for row in rows if predicted_refusal(row[prediction_field])
    ]
    correct_refusals = [row for row in refusal_predictions if not row["answerable"]]
    precision = len(correct_refusals) / max(len(refusal_predictions), 1)
    recall = len(correct_refusals) / max(len(unanswerable_rows), 1)
    no_answer_f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "strict_em": sum(score["exact"] for score in scored) / len(rows),
        "style_em": sum(score["style_exact"] for score in scored) / len(rows),
        "guarded_f1": sum(score["guarded_f1"] for score in scored) / len(rows),
        "relaxed_f1": sum(score["relaxed_f1"] for score in scored) / len(rows),
        "answerable_em": sum(score["exact"] for _, score in answerable_rows)
        / max(len(answerable_rows), 1),
        "unanswerable_em": sum(score["exact"] for _, score in unanswerable_rows)
        / max(len(unanswerable_rows), 1),
        "over_refusals": sum(
            predicted_refusal(row[prediction_field]) for row, _ in answerable_rows
        ),
        "false_answers": sum(
            bool(normalize_answer(row[prediction_field]))
            and not predicted_refusal(row[prediction_field])
            for row, _ in unanswerable_rows
        ),
        "empty_predictions": sum(
            not normalize_answer(row[prediction_field]) for row in rows
        ),
        "no_answer_precision": precision,
        "no_answer_recall": recall,
        "no_answer_f1": no_answer_f1,
    }


stage_metrics = {
    stage: aggregate_stage(predictions, field) for stage, field in STAGES.items()
}
summary_table = pd.DataFrame(stage_metrics).T
rate_columns = [
    "strict_em", "style_em", "guarded_f1", "relaxed_f1",
    "answerable_em", "unanswerable_em",
    "no_answer_precision", "no_answer_recall", "no_answer_f1",
]
display_table = summary_table.copy()
for column in rate_columns:
    display_table[column] = (100 * display_table[column]).map(lambda value: f"{value:.2f}%")

print("Raw vs repaired vs final metrics")
display(display_table)

quality = {
    "emitted_chat_eos": sum(row["emitted_eos"] for row in predictions),
    "emitted_document_end_before_chat_eos": sum(
        row["emitted_document_end"] for row in predictions
    ),
    "hit_token_cap": sum(row["hit_token_cap"] for row in predictions),
    "unhealthy_termination": sum(row["unhealthy_termination"] for row in predictions),
    "structural_trims": sum(row["structural_trimmed"] for row in predictions),
    "grounded_prefix_trims": sum(row["grounding_trimmed"] for row in predictions),
    "gate_eligible": sum(row["gate_eligible"] for row in predictions),
    "gated_to_refusal": sum(row["gated"] for row in predictions),
    "empty_generations": sum(row["empty_generation"] for row in predictions),
    "mean_generated_tokens": sum(row["generated_tokens"] for row in predictions)
    / len(predictions),
}

print("\nTermination and repair health")
for name, value in quality.items():
    print(f"  {name:38s}: {value:.2f}" if isinstance(value, float) else f"  {name:38s}: {value}")

summary_payload = {
    "runtime": runtime_config,
    "stage_metrics": stage_metrics,
    "quality": quality,
}
SUMMARY_JSON.write_text(
    json.dumps(summary_payload, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

summary_lines = [
    "",
    "=" * 100,
    "EXTERNAL TEST RESULTS — TRAINING-MATCHED QWEN PROMPT + RECORDED SAFETY POLICY",
    "=" * 100,
    f"Model                 : {MODEL_ID}",
    f"Model commit          : {model_commit}",
    f"Test file             : {TEST_PATH}",
    f"Test SHA256           : {dataset_sha256}",
    f"Rows                  : {len(predictions)}",
    f"Decoding              : {'beam x' + str(NUM_BEAMS) if NUM_BEAMS > 1 else 'greedy'}",
    f"Safety policy         : {GROUNDING_POLICY} (threshold={GROUNDING_THRESHOLD})",
]
for stage in ("raw", "repaired", "final"):
    metrics = stage_metrics[stage]
    summary_lines.extend([
        "",
        f"--- {stage.upper()} ---",
        f"Strict/style EM       : {100 * metrics['strict_em']:.2f}% / {100 * metrics['style_em']:.2f}%",
        f"Guarded/relaxed F1    : {metrics['guarded_f1']:.4f} / {metrics['relaxed_f1']:.4f}",
        f"Answerable/unans EM   : {100 * metrics['answerable_em']:.2f}% / {100 * metrics['unanswerable_em']:.2f}%",
        f"Over-refusal/false ans: {metrics['over_refusals']} / {metrics['false_answers']}",
        f"Empty predictions     : {metrics['empty_predictions']}",
        f"No-answer P/R/F1      : {metrics['no_answer_precision']:.4f} / {metrics['no_answer_recall']:.4f} / {metrics['no_answer_f1']:.4f}",
    ])
summary_lines.extend([
    "",
    f"Emitted <|im_end|>    : {quality['emitted_chat_eos']}/{len(predictions)}",
    f"Document end first    : {quality['emitted_document_end_before_chat_eos']}/{len(predictions)}",
    f"Hit token cap         : {quality['hit_token_cap']}/{len(predictions)}",
    f"Structural trims      : {quality['structural_trims']}/{len(predictions)}",
    f"Grounded-prefix trims : {quality['grounded_prefix_trims']}/{len(predictions)}",
    f"Gated to refusal      : {quality['gated_to_refusal']}/{len(predictions)}",
    "=" * 100,
])

transcript_lines.extend(summary_lines)
RESULTS_TXT.write_text("\n".join(transcript_lines) + "\n", encoding="utf-8")
INCOMPLETE_MARKER.unlink(missing_ok=True)
COMPLETE_MARKER.write_text(
    json.dumps({"completed_utc": datetime.now(timezone.utc).isoformat()}, indent=2) + "\n",
    encoding="utf-8",
)

print("\n" + "\n".join(summary_lines))
print("Saved transcript :", RESULTS_TXT)
print("Saved summary    :", SUMMARY_JSON)
print("Saved run config :", RUNTIME_CONFIG_JSON)
print("Completion marker:", COMPLETE_MARKER)


In [ ]:
results_df = pd.DataFrame(predictions)
results_df["repair_f1_delta"] = results_df["repaired_token_f1"] - results_df["raw_token_f1"]
results_df["final_f1_delta"] = results_df["token_f1"] - results_df["raw_token_f1"]


def show_cases(title, frame, columns=None, limit=10):
    print(f"\n{title}: {len(frame)}")
    if frame.empty:
        return
    default_columns = [
        "index", "answerable", "question", "reference",
        "raw_prediction", "repaired_prediction", "prediction",
        "evidence_support", "decision", "repair_reasons",
        "emitted_eos", "emitted_document_end", "hit_token_cap", "token_f1",
    ]
    display(frame[(columns or default_columns)].head(limit))


show_cases(
    "Did not emit the supervised <|im_end|>",
    results_df[~results_df["emitted_eos"]],
)
show_cases(
    "Emitted <|endoftext|> before chat EOS",
    results_df[results_df["emitted_document_end"]],
)
show_cases(
    "Hit the 64-token cap",
    results_df[results_df["hit_token_cap"]],
)
show_cases(
    "Repair changed output",
    results_df[
        results_df["structural_trimmed"] | results_df["grounding_trimmed"]
    ].sort_values("repair_f1_delta", ascending=False),
)
show_cases(
    "All answerable over-refusals",
    results_df[
        results_df["answerable"] & results_df["prediction"].map(is_no_answer)
    ],
)
show_cases(
    "False answers on unanswerable rows",
    results_df[
        (~results_df["answerable"])
        & (~results_df["prediction"].map(is_no_answer))
    ],
)
show_cases(
    "Accepted wrong-but-grounded answers (provenance is not relevance)",
    results_df[
        results_df["answerable"]
        & (results_df["decision"] == "accepted")
        & (results_df["token_f1"] < 0.5)
    ].sort_values("evidence_support", ascending=False),
)
show_cases(
    "Answerable rows below 0.5 final F1",
    results_df[results_df["answerable"] & (results_df["token_f1"] < 0.5)]
    .sort_values("token_f1"),
)
show_cases(
    "Repair reduced F1 (inspect for an over-aggressive heuristic)",
    results_df[results_df["repair_f1_delta"] < 0]
    .sort_values("repair_f1_delta"),
)


In [ ]:
# Convenience API for a new context/question after the evaluation run.
def answer_question(context, question, show_debug=True):
    result = run_qa(context, question)
    if show_debug:
        print("Raw      :", result["raw_answer"])
        print("Repaired :", result["repaired_answer"])
        print("Final    :", result["answer"])
        print(
            "Health   :",
            f"EOS={result['emitted_eos']} ({result['stop_token_id']}), "
            f"tokens={result['generated_tokens']}, cap={result['hit_token_cap']}, "
            f"support={format_support(result['evidence_support'])}, "
            f"decision={result['decision']}",
        )
    return result["answer"]


# Example:
# answer_question("මෙහි සන්දර්භය ...", "ප්‍රශ්නය ...?")


## Reading the result

- The learned QA stop is **only** `<|im_end|>`. `<|endoftext|>` remains the padding/document
  separator and is tracked as a failure signal, not accepted as successful termination.
- A healthy QA checkpoint should emit `<|im_end|>` on nearly every row and almost never hit
  the 64-token cap. Low chat-EOS with good final scores means the repair layer is masking a
  model-termination problem; report it as inference-time mitigation.
- Compare **raw**, **repaired**, and **final** sections in both `summary.json` and
  `results.txt`. Final scores must not be presented as raw model accuracy when repair or
  gating changed outputs.
- The default `unhealthy_only` policy preserves clean chat-EOS answers. Grounded-prefix
  trimming and the rejection gate can act only after unhealthy termination or an explicit
  contamination signal. This FINAL stage is therefore not directly comparable to older
  always-grounded reports; the RAW stage is the prompt/decoding comparison. Setting the
  policy to `always` restores unconditional prefix cleanup and gating, but should be done
  only with a threshold frozen on held-out training-context validation.
- Strict EM remains the comparable headline metric. Style-insensitive EM only removes a
  small set of Sinhala copula/particle variants; relaxed F1 can hide substantive errors, so
  both are diagnostics rather than replacements.
- The lexical support score verifies approximate provenance, not question relevance. A wrong
  name copied from another sentence in the same context can still pass. Remaining
  wrong-but-grounded answers require better QA supervision, an extractive reranker, or a new
  fine-tuning run—not more string cleanup.
- Each default run uses a new timestamped output directory and writes `RUN_COMPLETE.json`
  only after summaries are finalized. A remaining `RUN_INCOMPLETE` marks a partial run.
- Do not tune decoding, grounding, or cleanup rules on this external test set. Freeze them on
  held-out validation, then run this notebook once for the report.
